# Donor targeting: proxy labels, subgroup transfer, and routing

Combine model scores with distance constraints in a fictional nonprofit outreach scenario.

This walkthrough reads the exported results from the reproduced Python experiment. It does not train models in its cells. To regenerate the results, run the command below from the repository root, then rerun these cells.

```bash
python -m decision_analytics.donor
```

See [case README](README.md) for the full protocol and interpretation.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'pyproject.toml').exists() and (p / 'src/decision_analytics').exists())
CASE = ROOT / 'projects' / 'donor-routing'

## Training-only model selection

Each stage selects its estimator on an internal holdout from the source training data, then refits the selected estimator.

In [2]:
pd.read_csv(CASE / 'results/model_selection_validation.csv').round(4)

,stage,model,target,split,roc_auc,average_precision,brier_score
0,Income proxy,Logistic regression,income_gt50k,internal validation,0.9022,0.7698,0.1061
1,Income proxy,Random forest,income_gt50k,internal validation,0.8882,0.7353,0.1145
2,Income proxy,Gradient boosting,income_gt50k,internal validation,0.9201,0.8171,0.0952
3,Past donation,Logistic regression,if_donated_last_year,internal validation,0.7145,0.7973,0.2007
4,Past donation,Random forest,if_donated_last_year,internal validation,0.6496,0.7403,0.2326
5,Past donation,Gradient boosting,if_donated_last_year,internal validation,0.7100,0.7952,0.2023


## Known and cold prospects

The overall winner need not transfer best to cold prospects. The hybrid rule comes from the original course analysis, so this is a retrospective comparison.

In [3]:
pd.read_csv(CASE / 'results/future_donation_test.csv').round(4)

,score,group,n,future_donation_auc,future_donation_ap
0,Income proxy,All,18090,0.7298,0.5616
1,Income proxy,Known,9045,0.6519,0.6670
2,Income proxy,Cold,9045,0.6711,0.3057
3,Past donation,All,18090,0.7526,0.6244
4,Past donation,Known,9045,0.7047,0.7444
5,Past donation,Cold,9045,0.6357,0.2371
6,Hybrid heuristic,All,18090,0.8040,0.6536
7,Hybrid heuristic,Known,9045,0.7047,0.7444
8,Hybrid heuristic,Cold,9045,0.6711,0.3057


## Route baselines

All routes reserve enough distance to return to the depot. Income scores are not calibrated donation probabilities; score-based projections and historical simulated proceeds are kept separate.

In [4]:
pd.read_csv(CASE / 'results/routes_test.csv').round(3)

,strategy,budget_km,visited,distance_km,score_based_projected_net,donations,historical_simulated_net
0,Top scores first,15.0,23,14.812,3667.632,19,3536.879
1,Nearest first,15.0,134,14.992,8913.177,47,8580.078
2,Threshold + nearest,15.0,97,14.946,12022.971,52,9765.542


## Travel-budget sensitivity

Keep the score threshold fixed while changing the distance allowance.

In [5]:
pd.read_csv(CASE / 'results/budget_sensitivity.csv').round(3)

,strategy,budget_km,visited,distance_km,score_based_projected_net,donations,historical_simulated_net
0,Threshold + nearest,2.0,16,1.994,2022.996,11,2100.057
1,Threshold + nearest,5.0,42,4.808,5461.554,26,4941.920
2,Threshold + nearest,8.0,65,7.997,8224.363,38,7195.030
3,Threshold + nearest,10.0,77,9.987,9683.385,40,7515.131
4,Threshold + nearest,15.0,97,14.946,12022.971,52,9765.542
5,Threshold + nearest,20.0,105,19.616,12898.432,58,10878.840
6,Threshold + nearest,30.0,105,19.616,12898.432,58,10878.840


## Decision visualization

![Case results](figures/subgroups_and_route.svg)

## Limits

The 200-person sample and scenario outcomes do not establish real fundraising impact. Haversine distances ignore road networks and visit time. The inherited hybrid rule needs a separate future population and response-probability calibration.